In [ ]:
import random
from pathlib import Path
from statistics import mean
import numpy as np
import torch
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch import nn
import torchvision.transforms as transforms

import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import timm
from pytorch_datasets import ISIC2018, BreakHis, Papsmear
from pytorch_datasets import augmentations

from easyfsl.samplers import TaskSampler
from easyfsl.methods.utils import evaluate
from torch.utils.data import DataLoader
from easyfsl.datasets import MiniImageNet
from easyfsl.datasets.default_configs import default_transform

from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
import math

from torch.utils.tensorboard import SummaryWriter

### Initial setup

In [ ]:
random_seed = 0
np.random.seed(random_seed)
torch.manual_seed(random_seed)
random.seed(random_seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = "cuda"
IMAGENET_NORMALIZATION = {"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]}

In [ ]:
cutout_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4),
    transforms.RandomHorizontalFlip(),
    augmentations.Cutout(n_holes=1, length=64),
    transforms.ToTensor(),
    transforms.Normalize(**IMAGENET_NORMALIZATION),
])
simple_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
])

train_transform = default_transform(image_size=224, training=True)
test_transform  = default_transform(image_size=224, training=False)

In [ ]:
# Training Parameters
n_way = 2
n_shot = 3
n_query = 10

n_episodes = 40000
n_tasks_per_epoch = 500
n_validation_tasks = 200

n_workers = 6

cd = False
cd_base_dataset = 'IN'
cd_target_dataset = 'ISIC'

# Selecting augmentation
augments = [None, 'cutout', 'mixup', 'cutmix']
augmentation = augments[0]

if augmentation == 'cutout':
    transform = cutout_transform
else:
    transform = train_transform


# Selecting a dataset
datasets = {0: 'ISIC2018', 1: 'BreakHis', 2: 'Papsmear'}
dataset = datasets[1]  # Dataset ID

if dataset == 'ISIC2018':
    root = 'C:/Datasets/ISIC2018'
    train_set = ISIC2018(root=root, split='train',
                         training=True, transforms=transform)
    val_set = ISIC2018(root=root, split='validation',
                       training=False,)
elif dataset == 'BreakHis':
    root = 'C:/Datasets/BreaKHis_v1'
    magnification = 100

    train_set = BreakHis(root=root, magnification=magnification,
                         split='train', training=True, transforms=transform)
    val_set = BreakHis(root=root, magnification=magnification,
                       split='validation', training=False,)
elif dataset == 'Papsmear':
    train_root = r"C:\Datasets\smear2005\New database pictures\train"
    test_root = r"C:\Datasets\smear2005\New database pictures\test"

    train_set = Papsmear(root=train_root, training=True, transform=transform)
    val_set = Papsmear(root=test_root, training=False)

if cd:
    if cd_base_dataset == "ISIC":
        root = 'C:/Datasets/ISIC2018'
        train_set = ISIC2018(root=root, split='train',
                             training=True, transforms=transform)
    elif cd_base_dataset == 'IN':
        train_specs_file = r"E:\Datasets\ImageNet1k\mini_imagenet\train.csv"
        train_set = MiniImageNet(root=r"C:\Datasets\miniIN\train",
                                 specs_file=train_specs_file,
                                 training=True,
                                 transform=train_transform,)

    if cd_target_dataset == 'ISIC':
        root = 'C:/Datasets/ISIC2018'
        val_set = ISIC2018(root=root, split='validation',
                           training=False, transforms=transform)
    elif cd_target_dataset == 'Papsmear':
        test_root = r"C:\Datasets\smear2005\New database pictures\test"
        val_set = Papsmear(root=test_root, training=False, transform=transform)
    elif cd_target_dataset == 'BreakHis':
        root = 'C:/Datasets/BreaKHis_v1'
        magnification = 100
        val_set = BreakHis(root=root, magnification=magnification,
                           split='validation', training=False, transforms=transform)

# train_sampler = TaskSampler(
#     train_set, n_way=n_way, n_shot=n_shot, n_query=n_query, n_tasks=n_tasks_per_epoch
# )
# test_sampler = TaskSampler(
#     val_set, n_way=n_way, n_shot=n_shot, n_query=n_query, n_tasks=n_validation_tasks
# )

# # Finally, the DataLoader. We customize the collate_fn so that batches are delivered
# # in the shape: (support_images, support_labels, query_images, query_labels, class_ids)
# train_loader = DataLoader(
#     train_set,
#     batch_sampler=train_sampler,
#     num_workers=n_workers,
#     pin_memory=True,
#     collate_fn=train_sampler.episodic_collate_fn,
#     persistent_workers=True,
# )
# val_loader = DataLoader(
#     val_set,
#     batch_sampler=test_sampler,
#     num_workers=n_workers,
#     pin_memory=True,
#     persistent_workers=True,
#     collate_fn=test_sampler.episodic_collate_fn,
# )


In [ ]:
# labels = np.array(train_set.get_labels())

In [ ]:
# from sklearn.utils.class_weight import compute_class_weight

In [ ]:
# compute_class_weight(class_weight='balanced', classes=[0,1,2,3] , y= labels)

In [ ]:
class_weights_isic  = torch.tensor([0.35164057, 2.11837376, 2.14535942, 1, 1,  4.58706226, 1]).cuda()
class_weights_break = torch.tensor([1, 1.30538462, 1, 1, 0.37585825, 1.99647059, 1.52882883, 2.39014085]).cuda()
class_weights_papsmear = torch.tensor([1.125, 0.9271978, 1.15582192, 0.85659898]).cuda()

In [ ]:
class SE(nn.Module):
    def __init__(self, in_channels, reduction_ratio=4):
        super(SE, self).__init__()
        self.fc1 = nn.Linear(in_channels, in_channels // reduction_ratio)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(in_channels // reduction_ratio, in_channels)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.sigmoid(out)
        return x * out

In [ ]:
def show_tensor(tensor):
    display(transforms.ToPILImage()(tensor[0]))
    print(tensor[1])

## Experiments

In [ ]:
# d = iter(train_loader)

In [ ]:
# support_images, support_labels, query_images, query_labels, _ = d.next()

## Training

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
# timm.list_models()

### Pre-train for transfer learning

In [ ]:
import copy

batch_size = 128

train_size = int(0.98 * len(train_set))
val_size = len(train_set) - train_size
trainset, valset = torch.utils.data.random_split(train_set, [train_size, val_size])
valset = copy.deepcopy(valset)

valset.dataset.transform   = val_set.transforms

train_loader = DataLoader(
    trainset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=n_workers,
    pin_memory=True,
    persistent_workers=True,                                     
)
val_loader = DataLoader(
    valset, 
    batch_size=batch_size, 
    shuffle=False, 
    num_workers=n_workers, 
    pin_memory=True,
    persistent_workers=True,
)

In [ ]:
lr = 0.03
# lrs = [0.03]
model_name = 'resnet50'
mags = [200]
# model_name = 'vit_small_patch16_224'

In [ ]:
device = DEVICE

if dataset == "ISIC2018":
    weights = class_weights_isic
    n_classes = 7
elif dataset == 'BreakHis':
    weights = class_weights_break
    n_classes = 8
elif dataset == 'Papsmear':
    weights = class_weights_papsmear
    n_classes = 4

for mag in mags:
    root = 'C:/Datasets/BreaKHis_v1'
    magnification = mag

    train_set = BreakHis(root=root, magnification=magnification,
                         split='train', training=True, transforms=transform)
    val_set = BreakHis(root=root, magnification=magnification,
                       split='validation', training=False,)
    
    batch_size = 128

    train_size = int(0.98 * len(train_set))
    val_size = len(train_set) - train_size
    trainset, valset = torch.utils.data.random_split(train_set, [train_size, val_size])
    valset = copy.deepcopy(valset)

    valset.dataset.transform   = val_set.transforms

    train_loader = DataLoader(
        trainset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=n_workers,
        pin_memory=True,
        persistent_workers=True,                                     
    )
    val_loader = DataLoader(
        valset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=n_workers, 
        pin_memory=True,
        persistent_workers=True,
    )

    model = timm.create_model(model_name, pretrained=True, num_classes=n_classes).cuda()

    warmup_steps = 100
    steps = 500
    num_epochs = math.ceil(steps / len(train_loader))
    
    loss_fn = nn.CrossEntropyLoss(weight=weights)
    # optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-5)
    # optimizer = optim.SGD(model.parameters(), lr=1e-3, momentum=0.9, weight_decay=0.1)
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)

    # Define loss function and optimizer
    warmup_scheduler = LinearLR(
        optimizer,
        start_factor=1/warmup_steps,
        end_factor=1, 
        total_iters=warmup_steps
    )
    cosine_scheduler = CosineAnnealingLR(
        optimizer,
        T_max=steps-warmup_steps,
        eta_min=0
    )

    scheduler = SequentialLR(
        optimizer, 
        [warmup_scheduler, cosine_scheduler],
        milestones=[warmup_steps]
    )

    writer = SummaryWriter(comment=f'{dataset}_{model_name}_mag{mag}_bs{batch_size}_lr{lr}nogc')
    # writer.add_hparams(hparam_dict=hparams, metric_dict={'acc': 0})

    #@title training steps
    best_model = None
    best_val_loss = float('inf')

    use_amp = True
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    pbar = tqdm(total=steps, smoothing = 0)
    step = 1

    tb_write = True

    for epoch in range(num_epochs):
        for (inputs, labels) in train_loader:
            model.train()

            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
                outputs = model(inputs)
                loss    = loss_fn(outputs, labels)
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)

            # nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            scaler.step(optimizer)
            scaler.update()
            # Update the learning rate scheduler
            scheduler.step()
            
            if tb_write:
                writer.add_scalar("Loss/train", loss.item(), step)
                writer.add_scalar("lr", scheduler.get_last_lr()[0], step)

            if step % 10 == 0:
                # Evaluation phase
                model.eval()
                val_loss = 0.0
                val_correct = 0
                
                with torch.no_grad():
                    for inputs, labels in val_loader:
                        inputs, labels = inputs.to(device), labels.to(device)

                        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
                            outputs = model(inputs)
                            loss = loss_fn(outputs, labels)

                        val_loss += loss.item() * inputs.size(0)
                        val_correct += torch.sum(torch.argmax(outputs, dim=1) == labels)

                    val_loss /= len(val_loader.dataset)
                    val_accuracy = val_correct.float() / len(val_loader.dataset)
                
                    if tb_write:
                        writer.add_scalar("Loss/test", val_loss, step)
                        writer.add_scalar("acc", val_accuracy, step)
            
            pbar.update(1)
            if(step == steps):
                break
            step += 1
            # if val_loss < best_val_loss:
            #     best_val_loss = val_loss
            #     best_model = model.state_dict()
            #     print('!!! New best model found !!!')
            #     # torch.save(best_model, 'best_model.pth')

    torch.save(model, f'E:/Models/{model_name}_{dataset}_mag{mag}_bs{batch_size}_lr{lr}.pth')

In [ ]:
# torch.save(model, f'E:/Models/{model_name}_{dataset}_bs{batch_size}_lr{lr}.pth')

In [ ]:
# model = torch.load(r"E:\Models\vit_small_patch16_224_BreakHis_bs128_lr0.03.pth")

## Evaluation

In [ ]:
for param in model.parameters():
    param.requires_grad = False

In [ ]:
inner_loops = [10, 50]
inner_lrs   = [0.01, 0.001, 0.0001]

In [ ]:
num_inner_loop_eval = 40
inner_loop_lr       = 0.01

criterion = nn.CrossEntropyLoss()

In [ ]:
for mag in mags:
    if dataset == 'Papsmear':
        model = torch.load(f"E:/Models/{model_name}_{dataset}_bs128_lr0.01.pth")
    else:
        model = torch.load(f'E:/Models/{model_name}_{dataset}_mag{mag}_bs{batch_size}_lr{lr}.pth')
    
    if dataset == 'ISIC2018':
        root = 'C:/Datasets/ISIC2018'
        val_set = ISIC2018(root=root, split='validation',
                        training=False,)
    elif dataset == 'BreakHis':
        root = 'C:/Datasets/BreaKHis_v1'
        magnification = mag
        val_set = BreakHis(root=root, magnification=magnification,
                        split='validation', training=False,)
    elif dataset == 'Papsmear':
        test_root = r"C:\Datasets\smear2005\New database pictures\test"

        val_set = Papsmear(root=test_root, training=False)

    for param in model.parameters():
        param.requires_grad = False

    test_batch_size = 200
    n_ways  = [2, 3]
    n_shots = [3, 5, 10]

    for n_way in n_ways:
        for number_shot in n_shots:
            test_sampler = TaskSampler(
                val_set, n_way=n_way, n_shot=number_shot, n_query=n_query, n_tasks=test_batch_size
            )

            test_loader = DataLoader(
                val_set,
                batch_sampler=test_sampler,
                num_workers=n_workers,
                pin_memory=True,
                collate_fn=test_sampler.episodic_collate_fn,
            )

            model.eval()
            total_accuracy = 0
            num_eval_tasks = len(test_loader)

            with tqdm(
                    enumerate(test_loader), total=len(test_loader), desc="eval loop", position=0, leave=False,
                ) as tqdm_val:
                    for episode_index, (
                        support_images,
                        support_labels,
                        query_images,
                        query_labels,
                        _,
                    ) in tqdm_val:

                        model.fc = nn.Linear(2048, n_way).cuda()
                        # model.head = nn.Linear(384, n_way).cuda()
                        task_optimizer = optim.SGD(filter(lambda p: p.requires_grad, model.parameters()), lr=inner_loop_lr)

                        for inner_iter in range(num_inner_loop_eval):
                            data, target = support_images.to(DEVICE), support_labels.to(DEVICE)

                            task_optimizer.zero_grad()
                            
                            preds = model(data)

                            loss = criterion(preds, target)
                            loss.backward()

                            task_optimizer.step()

                        # Test the adapted model on the query set
                        data, target = query_images.to(DEVICE), query_labels.to(DEVICE)
                        preds = model(data)
                        pred_labels = preds.argmax(dim=1)
                        accuracy = (pred_labels == target).float().mean().item()
                        total_accuracy += accuracy
                        del preds

            mean_accuracy = total_accuracy / num_eval_tasks
            print(f"{dataset}_mag{mag}_{num_inner_loop_eval}_lr{inner_loop_lr}_{model_name}-{augmentation} {n_way}-way {number_shot}-shot Test accuracy: {mean_accuracy:.4f}")
            # reptile_tqdm.set_postfix(test_accuracy=mean_accuracy)

In [ ]:
# torch.save(few_shot_classifier, f'E:/Models/protonet_checkpoints/{model_name}.pt')